In [1]:
# ============================================
# 0) IMPORTS
# ============================================
import os
import glob
import pandas as pd
import numpy as np
import shapely.wkt
import shapely.geometry
import geopandas as gpd
from affine import Affine
import rasterio.features
from tqdm import tqdm  # Progress bar
import gc  # Garbage Collector for memory management

import ee

# ============================================
# 1) CONFIGURATION
# ============================================

# --- INPUT: The consolidated PKL from step 02a ---
INPUT_PKL = r"D:\Development\RESEARCH\urban_flood_database\chronicle\hydromerit_pluvial_outputs\chronicle_df_with_pfdi_FULL.pkl"

# --- OUTPUT: Where rain data will be saved (Updated to network path) ---
OUT_DIR = r"\\vscifs\hydrolab1\hydrolab\home\Raz\Research\chronicle"
OUT_FINAL_PKL = os.path.join(OUT_DIR, "chronicle_urban_df_with_IMERG_FULL.pkl")

# IMERG Constants (Updated to V07)
# Data available from June 2000
IMERG_START_DATE = pd.Timestamp("2000-06-01") 
SCALE = 0.1  # 0.1 Degree resolution
CRS = 'EPSG:4326'

# Batch Settings
BATCH_SIZE = 1000 
N_BATCHES_TO_RUN = 500  

# ============================================
# 2) HELPERS
# ============================================

def ensure_out_dir(path):
    """Create output directory if it doesn't exist."""
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

def initialize_ee():
    """Initialize Earth Engine."""
    try:
        ee.Initialize()
        print("Earth Engine initialized.")
    except Exception:
        print("Authenticating Earth Engine...")
        ee.Authenticate()
        ee.Initialize()
        print("Earth Engine initialized after auth.")

def get_next_batch_index(out_dir):
    """
    Scans output directory for 'imerg_batch_XXXX.pkl' to determine 
    the next batch number for file naming.
    """
    if not os.path.exists(out_dir):
        return 0
    
    pattern = os.path.join(out_dir, "imerg_batch_*.pkl")
    files = glob.glob(pattern)
    
    if not files:
        return 0
    
    max_batch = -1
    for f in files:
        try:
            filename = os.path.basename(f)
            num_part = filename.split('_')[-1].split('.')[0]
            batch_num = int(num_part)
            if batch_num > max_batch:
                max_batch = batch_num
        except ValueError:
            continue
            
    return max_batch + 1

def extract_rain_data(event_row):
    """
    Extracts IMERG rain matrix, metadata, and mask for a single event.
    Updated for V07: Band name changed from 'precipitationCal' to 'precipitation'.
    """
    try:
        # 1. Geometry Setup
        if isinstance(event_row['geometry_wkt'], str):
            poly_geom = shapely.wkt.loads(event_row['geometry_wkt'])
        else:
            poly_geom = event_row['geometry_wkt']
            
        bounds = poly_geom.bounds 
        roi = ee.Geometry.BBox(*bounds)
        
        # 2. Time Window Calculation
        start_t = event_row['start_time'] - pd.Timedelta(hours=72)
        end_t = event_row['end_time'] + pd.Timedelta(hours=24) 
        
        # 3. GEE Request (Updated band name to 'precipitation')
        imerg_type = 'final'
        imerg_coll = ee.ImageCollection("NASA/GPM_L3/IMERG_V07") \
            .select('precipitation') \
            .filterBounds(roi) \
            .filterDate(start_t, end_t)
        
        if imerg_coll.size().getInfo() == 0:
            imerg_type = 'late'
            imerg_coll = ee.ImageCollection("NASA/GPM_L3/IMERG_V07_LATE") \
                .select('precipitation') \
                .filterBounds(roi) \
                .filterDate(start_t, end_t)

        if imerg_coll.size().getInfo() == 0:
            return None, None, None, 'missing'

        # 4. Download Data
        stack = imerg_coll.toBands()
        pixel_dict = stack.sampleRectangle(region=roi, defaultValue=0).getInfo()
        properties = pixel_dict['properties']
        
        # 5. Parse & Stack Arrays
        band_keys = sorted(list(properties.keys()))
        arrays_list = [np.array(properties[b], dtype=np.float32) for b in band_keys]
        rain_matrix = np.stack(arrays_list)
        
        # 6. Create Metadata
        height, width = rain_matrix.shape[1], rain_matrix.shape[2]
        min_lon, min_lat, max_lon, max_lat = bounds
        transform = Affine(SCALE, 0, min_lon, 0, -SCALE, max_lat)
        
        meta = {
            'origin_top_left': (max_lat, min_lon),
            'scale': SCALE,
            'shape': (height, width),
            'timestamps': band_keys
        }

        # 7. Create Binary Mask
        mask = rasterio.features.rasterize(
            [(poly_geom, 1)],
            out_shape=(height, width),
            transform=transform,
            fill=0,
            all_touched=True,
            dtype=np.uint8
        )
        
        return rain_matrix, mask, meta, imerg_type

    except Exception as e:
        print(f"Error processing ID {event_row.get('event_id')}: {e}")
        return None, None, None, 'error'

In [ ]:
# ============================================
# 3) INITIALIZATION
# ============================================
initialize_ee()
ensure_out_dir(OUT_DIR)

print(f"Loading full dataset: {INPUT_PKL}")
if not os.path.exists(INPUT_PKL):
    raise FileNotFoundError(f"Input file not found: {INPUT_PKL}.")

df = pd.read_pickle(INPUT_PKL)
df = df.replace([np.inf, -np.inf], np.nan)

# Ensure datetime format
if not pd.api.types.is_datetime64_any_dtype(df['start_time']):
    df['start_time'] = pd.to_datetime(df['start_time'], unit='s')
if not pd.api.types.is_datetime64_any_dtype(df['end_time']):
    df['end_time'] = pd.to_datetime(df['end_time'], unit='s')

# Filter for IMERG Era (Data from June 2000 onwards)
df_valid = df[df['start_time'] >= IMERG_START_DATE].copy()

print(f"Total events in input: {len(df)}")
print(f"Events valid for IMERG V07 (post-2000): {len(df_valid)}")

# ============================================
# 4) SMART BATCH PROCESSING (ID-BASED)
# ============================================

print(f"--- PREPARING WORK PLAN ---")

processed_ids = set()
pkl_pattern = os.path.join(OUT_DIR, "imerg_batch_*.pkl")
existing_files = glob.glob(pkl_pattern)

if existing_files:
    print(f"Found {len(existing_files)} existing batch files in network path. Scanning...")
    for f in tqdm(existing_files, desc="Indexing existing data"):
        try:
            df_temp = pd.read_pickle(f)
            if 'event_id' in df_temp.columns:
                # Add IDs to set for O(1) lookup
                processed_ids.update(df_temp['event_id'].tolist())
            del df_temp
        except Exception as e:
            print(f"Skipping corrupted file {f}: {e}")

print(f"Total events already processed: {len(processed_ids)}")

# Filter out what's already done
df_todo = df_valid[~df_valid['event_id'].isin(processed_ids)].copy()
print(f"Events remaining to process: {len(df_todo)}")

if len(df_todo) == 0:
    print("All events are already processed! Nothing to do.")
else:
    start_batch_num = get_next_batch_index(OUT_DIR)
    n_remaining = len(df_todo)
    max_rows_limit = N_BATCHES_TO_RUN * BATCH_SIZE
    stop_at_row = min(n_remaining, max_rows_limit)

    print(f"Plan: Processing {min(N_BATCHES_TO_RUN, (stop_at_row // BATCH_SIZE) + 1)} batches.")
    
    for batch_i in range(0, stop_at_row, BATCH_SIZE):
        current_file_num = start_batch_num + (batch_i // BATCH_SIZE)
        batch_df = df_todo.iloc[batch_i : batch_i + BATCH_SIZE].copy()
        
        print(f"\nProcessing Batch {current_file_num} ({len(batch_df)} events)...")
        
        matrices, masks, metas, types = [], [], [], []
        
        for idx, row in tqdm(batch_df.iterrows(), total=len(batch_df), desc=f"Batch {current_file_num}"):
            # Calls the updated V07 extraction function
            mat, msk, mt, itype = extract_rain_data(row)
            matrices.append(mat)
            masks.append(msk)
            metas.append(mt)
            types.append(itype)
        
        # Assign results to the batch dataframe
        batch_df['imerg_matrix'] = matrices
        batch_df['imerg_mask'] = masks
        batch_df['imerg_meta'] = metas
        batch_df['imerg_type'] = types
        
        # Save with network error handling
        out_path = os.path.join(OUT_DIR, f"imerg_batch_{current_file_num:04d}.pkl")
        try:
            batch_df.to_pickle(out_path)
            print(f"Successfully saved to network: {out_path}")
        except Exception as e:
            print(f"CRITICAL ERROR: Could not save {out_path}. Network might be down. Error: {e}")
            # Optional: break or continue depending on your preference
        
        # Clear memory
        del batch_df, matrices, masks, metas, types
        gc.collect()

    print("\n--- BATCH LIMIT REACHED OR PROCESSING FINISHED ---")

Earth Engine initialized.
Loading full dataset: D:\Development\RESEARCH\urban_flood_database\chronicle\hydromerit_pluvial_outputs\chronicle_df_with_pfdi_FULL.pkl
Total events in input: 882957
Events valid for IMERG V07 (post-2000): 882661
--- PREPARING WORK PLAN ---
Found 275 existing batch files in network path. Scanning...


Indexing existing data: 100%|████████████████████████████████████████████████████| 275/275 [01:09<00:00,  3.98it/s]


Total events already processed: 275000
Events remaining to process: 607661
Plan: Processing 500 batches.

Processing Batch 275 (1000 events)...


Batch 275: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [14:37<00:00,  1.14it/s]


Successfully saved to network: \\vscifs\hydrolab1\hydrolab\home\Raz\Research\chronicle\imerg_batch_0275.pkl

Processing Batch 276 (1000 events)...


...

Batch 346: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [15:00<00:00,  1.11it/s]


Successfully saved to network: \\vscifs\hydrolab1\hydrolab\home\Raz\Research\chronicle\imerg_batch_0346.pkl

Processing Batch 347 (1000 events)...


Batch 347: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [13:58<00:00,  1.19it/s]


Successfully saved to network: \\vscifs\hydrolab1\hydrolab\home\Raz\Research\chronicle\imerg_batch_0347.pkl

Processing Batch 348 (1000 events)...


Batch 348:   9%|█████▌                                                           | 86/1000 [03:21<29:57,  1.97s/it]

In [ ]:
# ============================================
# 5) FINAL MERGE & CLEANUP (V07 Version)
# ============================================
print("\n--- FINALIZING V07 DATASET ---")
print(f"Merging batches from: {OUT_DIR}")

pkl_pattern = os.path.join(OUT_DIR, "imerg_batch_*.pkl")
all_pkl_files = glob.glob(pkl_pattern)

if not all_pkl_files:
    print("No output files found. Check your OUT_DIR path.")
else:
    df_list = []
    for f in tqdm(all_pkl_files, desc="Loading Batches"):
        try:
            # Code comments must be in English only as per your preference
            df_list.append(pd.read_pickle(f))
        except Exception as e:
            print(f"Error loading {f}: {e}")
            
    if df_list:
        df_results = pd.concat(df_list, ignore_index=True)
        
        # Define relevant columns for V07
        merge_cols = ['event_id', 'imerg_matrix', 'imerg_mask', 'imerg_meta']
        if 'imerg_type' in df_results.columns:
            merge_cols.append('imerg_type')
        
        # Load base dataset to merge with extracted rain data
        df_base = pd.read_pickle(INPUT_PKL)
        
        df_final = df_base.merge(
            df_results[merge_cols], 
            on='event_id', 
            how='inner' 
        )
        
        # Tag data as V07 for future tracking
        if 'imerg_type' in df_final.columns:
            df_final['imerg_type'] = df_final['imerg_type'].fillna('V07_final')
        else:
            df_final['imerg_type'] = 'V07_final'

        # Cleanup and save
        missing_rain_mask = df_final['imerg_matrix'].isnull()
        df_final = df_final[~missing_rain_mask].copy()
        
        df_final.to_pickle(OUT_FINAL_PKL)
        print(f"SUCCESS! Final V07 dataset saved to: {OUT_FINAL_PKL}")

In [6]:
df_final

,Unnamed: 0,uuid,area_km2,version,start_time,end_time,duration_days,geometry_wkt,urban_built_up_area_m2,polygon_total_area_m2,...,upa_max,upa_p95,upa_p99,PFDI_p95,PFDI_p99,PFDI_max,imerg_matrix,imerg_mask,imerg_meta,imerg_type
0,296,1422c1c0986549d7a3c7d10a09208098,18.198011,v3.1,9.598176e+08,9.598176e+08,1,"POLYGON ((-98.923354 19.259108, -98.916374 19....",6055133.0,1.790232e+07,...,7775.601562,37.711626,7437.391806,2.117765,417.660235,436.653018,"[[[0.17999999, 0.0]], [[0.26, 0.0]], [[0.53999...","[[1, 0]]","{'origin_top_left': (19.288329, -98.923354), '...",final
1,297,43553235080041299c3e48978885c635,38.654336,v3.1,9.598176e+08,9.598176e+08,1,"POLYGON ((-98.953929 19.329468, -98.883774 19....",8815066.0,2.984955e+07,...,7266.895020,0.360294,42.387838,0.012133,1.427476,244.723996,"[[[0.17, 0.0], [0.17999999, 0.0]], [[0.2699999...","[[1, 0], [0, 0]]","{'origin_top_left': (19.346044, -98.953929), '...",final
2,298,6532dc29aaa645239f52ad164489e0aa,1678.933557,v3.1,9.598176e+08,9.598176e+08,1,"POLYGON ((95.067367 28.023449, 95.288819 28.09...",6223800.0,1.760002e+09,...,255259.687500,1.637564,1.637564,0.000933,0.000933,145.503978,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.07, 0.0, ...","[[0, 1, 1, 1, 1, 0], [0, 1, 1, 1, 1, 0], [1, 1...","{'origin_top_left': (28.320151, 95.067367), 's...",final
3,299,bd7267cb286448798c31cfab0f6a0f95,28.562733,v3.1,9.598176e+08,9.598176e+08,1,"POLYGON ((-98.97432499999999 19.337026, -98.93...",3757833.0,1.180029e+07,...,8171.314453,0.613581,0.613581,0.052305,0.052305,696.567534,"[[[0.17], [0.17999999]], [[0.26999998], [0.26]...","[[1], [0]]","{'origin_top_left': (19.337026, -98.974325), '...",final
4,300,d2205ca5dd1c42f2b6ca67ab9bf1e9e8,117.791321,v3.1,9.598176e+08,9.598176e+08,1,"POLYGON ((1.3503293 43.604335, 1.4399303 43.66...",28337179.0,1.056523e+08,...,10537.763672,0.304269,648.954102,0.002881,6.145526,99.791494,"[[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0....","[[1, 1, 0], [1, 1, 0]]","{'origin_top_left': (43.668714, 1.3503293), 's...",final
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
274995,275294,e39fb768e44d4237996d22e51b23e8d3,2.722378,v3.1,1.534291e+09,1.534291e+09,1,"POLYGON ((76.189156 10.388039, 76.179543 10.40...",154870.0,2.956368e+06,...,791.573730,0.041140,790.778885,0.014004,269.170530,269.441084,"[[[1.4499999], [1.15]], [[1.8499999], [2.07]],...","[[1], [0]]","{'origin_top_left': (10.41193, 76.179543), 'sc...",final
274996,275295,e36e85be608549ed89de8fdc7b784dfd,44.558569,v3.1,1.534291e+09,1.534291e+09,1,"POLYGON ((21.69469 49.708474, 21.786936 49.711...",4004669.0,3.631958e+07,...,686.779297,0.068976,558.598897,0.001897,15.361519,18.886491,"[[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0....","[[1, 1, 0], [0, 0, 0]]","{'origin_top_left': (49.711826, 21.69469), 'sc...",final
274997,275296,e407c25150204fc2b41499eddfbb1314,1.592890,v3.1,1.534291e+09,1.534378e+09,2,"POLYGON ((76.730576 9.2239039, 76.734545 9.233...",62196.0,1.092868e+06,...,776.791687,0.129817,776.708166,0.119545,715.251729,715.328641,"[[[0.0]], [[0.0]], [[0.0]], [[0.0]], [[0.28]],...",[[1]],"{'origin_top_left': (9.2333079, 76.730576), 's...",final
274998,275297,e2ae865a60b544daa798be4de86fdc38,70.526543,v3.1,1.534291e+09,1.534291e+09,1,"POLYGON ((95.21065299999999 18.821974, 95.2242...",3298080.0,5.139447e+07,...,1939.907959,0.188881,1850.437447,0.003694,36.191904,37.941819,"[[[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, ...","[[1, 0], [0, 0]]","{'origin_top_left': (18.862359, 95.210653), 's...",final


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 1. SETUP: Style & Theme
try:
    if 'seaborn-v0_8-whitegrid' in plt.style.available:
        plt.style.use('seaborn-v0_8-whitegrid')
    elif 'seaborn-whitegrid' in plt.style.available:
        plt.style.use('seaborn-whitegrid')
    else:
        plt.style.use('bmh')
except:
    plt.style.use('default')

sns.set_theme(style="whitegrid")

# 2. PREPARE DATA
df_plot = df_final[df_final['imerg_matrix'].notnull()].copy()

if df_plot.empty:
    print("No valid rain data found in df_final.")
else:
    # Pre-calculate peak intensity for each event
    df_plot['peak_intensity'] = df_plot['imerg_matrix'].apply(lambda x: np.max(x))
    
    fig, axes = plt.subplots(3, 2, figsize=(16, 20))
    fig.suptitle(f"IMERG V07 Analysis - {len(df_plot)} Events (Variable Shapes)", fontsize=22, y=1.02)

    # --- GRAPH 1: Average Spatial Rainfall (Centralized Normalized) ---
    # Since shapes differ, we take a sample of the first valid matrix for visualization
    # Or calculate the mean of the MAX value across all events for a spatial feel
    sample_idx = len(df_plot) // 2
    sample_matrix = df_plot['imerg_matrix'].iloc[sample_idx]
    mean_spatial_sample = np.mean(sample_matrix, axis=0)
    
    im1 = axes[0, 0].imshow(mean_spatial_sample, cmap='YlGnBu', interpolation='nearest')
    axes[0, 0].set_title(f"1. Spatial Distribution Example (ID: {df_plot['event_id'].iloc[sample_idx]})", fontsize=14, fontweight='bold')
    plt.colorbar(im1, ax=axes[0, 0], label='Intensity (mm/hr)')

    # --- GRAPH 2: Mean Event Hyetograph (Handles different shapes) ---
    # We calculate the mean time-series for EACH event first, then average them
    # This works as long as the temporal dimension (axis 0) is the same (usually 192 steps)
    all_series = []
    for m in df_plot['imerg_matrix']:
        all_series.append(np.mean(m, axis=(1, 2))) # Mean over space for this event
    
    all_series_stacked = np.stack(all_series)
    final_mean_series = np.mean(all_series_stacked, axis=0)
    
    axes[0, 1].plot(final_mean_series, color='#2c7bb6', lw=2.5)
    axes[0, 1].fill_between(range(len(final_mean_series)), final_mean_series, color='#abd9e9', alpha=0.5)
    axes[0, 1].set_title("2. Mean Temporal Profile (Hyetograph)", fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel("Time Steps (30-min)")
    axes[0, 1].set_ylabel("Avg Intensity (mm/hr)")

    # --- GRAPH 3: Distribution of Peak Intensities ---
    sns.histplot(df_plot['peak_intensity'], bins=40, kde=True, ax=axes[1, 0], color='#d7191c')
    axes[1, 0].set_title("3. Distribution of Event Peak Intensities", fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel("Max Intensity Observed (mm/hr)")
    axes[1, 0].set_ylabel("Number of Events")

    # --- GRAPH 4: IMERG Quality Distribution (V07) ---
    if 'imerg_type' in df_plot.columns:
        type_counts = df_plot['imerg_type'].value_counts()
        axes[1, 1].pie(type_counts, labels=type_counts.index, autopct='%1.1f%%', 
                       colors=['#4daf4a','#377eb8'], startangle=90)
        axes[1, 1].set_title("4. Data Quality (V07 Source)", fontsize=14, fontweight='bold')

    # --- GRAPH 5: Seasonality ---
    df_plot['month'] = df_plot['start_time'].dt.month
    monthly_avg = df_plot.groupby('month')['peak_intensity'].mean().reset_index()
    
    sns.barplot(data=monthly_avg, x='month', y='peak_intensity', ax=axes[2, 0], palette='crest')
    axes[2, 0].set_title("5. Monthly Mean Peak Intensity", fontsize=14, fontweight='bold')
    axes[2, 0].set_xlabel("Month")
    axes[2, 0].set_ylabel("Mean Peak (mm/hr)")

    # --- Info Box ---
    axes[2, 1].axis('off')
    info_text = (f"V07 Data Summary\n"
                 f"Total Events: {len(df_plot)}\n"
                 f"Max Intensity: {df_plot['peak_intensity'].max():.2f} mm/hr")
    axes[2, 1].text(0.5, 0.5, info_text, ha='center', va='center', fontsize=14, 
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))

    plt.tight_layout()
    plt.show()

In [ ]:
# Check the distribution of IMERG data quality
print(df_final['imerg_type'].value_counts())

# HERE SHOULD BE COMPLEMENTERY CODE:
הקוד צריך לעבור שוב על כל האירועים שהוסרו מהבאצים כי לא היה בהם דאטא.
הוא יבחן שוב אם יש את הדאטא שלהם בגרסה הטובה ביותר ואם לא אז בגרסה השנייה הפחות טובה.
הרעיון - לודא שלא נפלו האירועים בגלל מגבלות גישה ועומס על הענן של גוגל בשלב ראשון, ואם לא זה אז האם יש בעיה בזמינות הנתונים עצמם בגרסה המירבית ואז זה יגש לגרסה הפחות טובה.


המלצת הגימיני:

# --- COMPLEMENTARY CLEANUP STEP ---
print("\n--- RUNNING COMPLEMENTARY RECOVERY ---")

# Load the reference (all events) and the current results
df_base = pd.read_pickle(INPUT_PKL)
df_final = pd.read_pickle(OUT_FINAL_PKL)

# Find IDs that are in base but NOT in final
missing_mask = ~df_base['event_id'].isin(df_final['event_id'])
df_missing = df_base[missing_mask].copy()

print(f"Status check: {len(df_missing)} events are currently missing from the final PKL.")

if len(df_missing) > 0:
    print(f"Attempting to recover {len(df_missing)} events...")
    recovered_list = []
    
    # Try one last time for each missing event
    for idx, row in tqdm(df_missing.iterrows(), total=len(df_missing), desc="Recovery"):
        # This will use the Fallback logic (Final -> Late)
        mat, msk, mt, itype = extract_rain_data(row)
        
        # Only keep if we actually found data this time
        if mat is not None:
            recovered_row = row.to_dict()
            recovered_row.update({
                'imerg_matrix': mat,
                'imerg_mask': msk,
                'imerg_meta': mt,
                'imerg_type': itype
            })
            recovered_list.append(recovered_row)
    
    if recovered_list:
        df_recovered = pd.DataFrame(recovered_list)
        # Combine the old final results with the newly recovered ones
        df_updated = pd.concat([df_final, df_recovered], ignore_index=True)
        
        # Final save
        df_updated.to_pickle(OUT_FINAL_PKL)
        print(f"Recovery successful! Added {len(recovered_list)} events.")
        print(f"Final count: {len(df_updated)} events.")
    else:
        print("No additional data could be recovered (all remaining events likely out of range or truly empty).")

In [ ]:
df_final

# לעדכן לגרסה V07 של imerg

In [ ]:
700912

In [8]:
print(ee.ImageCollection("NASA/GPM_L3/IMERG_V06").limit(1, "system:time_start", False).first().date().format().getInfo())

2024-06-02T18:30:00


In [13]:
print(ee.ImageCollection("NASA/GPM_L3/IMERG_V07").limit(1, "system:time_start", False).first().date().format().getInfo())

2026-02-14T03:30:00
